In [ ]:
!pip install mediapipe opencv-python speechrecognition pyaudio numpy python-osc

In [ ]:
import cv2
import mediapipe as mp
import threading
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import clear_output, display

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(min_detection_confidence=0.7)
mp_drawing = mp.solutions.drawing_utils

# Variable compartida
gesto_actual = "ninguno"

def detectar_mano():
    global gesto_actual
    cap = cv2.VideoCapture(0)
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(image)

        gesto_actual = "ninguno"

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

                # Gesto simple: mano abierta (pulgar arriba)
                y_dedo_pulgar = hand_landmarks.landmark[mp_hands.HandLandmark.THUMB_TIP].y
                y_dedo_indice = hand_landmarks.landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP].y

                if y_dedo_indice < y_dedo_pulgar:
                    gesto_actual = "mano_abierta"
                else:
                    gesto_actual = "dos_dedos"

        cv2.imshow('MediaPipe Manos', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

In [ ]:
import speech_recognition as sr

comando_actual = ""

def escuchar_comandos():
    global comando_actual
    r = sr.Recognizer()
    mic = sr.Microphone()

    with mic as source:
        r.adjust_for_ambient_noise(source)

    while True:
        with mic as source:
            print("🎤 Escuchando...")
            audio = r.listen(source)
        try:
            comando = r.recognize_google(audio, language="es-ES")
            print("Comando reconocido:", comando)
            comando_actual = comando.lower()
        except sr.UnknownValueError:
            print("No se entendió el audio.")

In [ ]:
import pygame
import numpy as np

def mostrar_accion(color_rgb, accion, gesto, comando):
    clear_output(wait=True)
    plt.figure(figsize=(4, 4))
    plt.gca().add_patch(Rectangle((0, 0), 1, 1, color=color_rgb))
    plt.xticks([])
    plt.yticks([])
    plt.title(f"Acción: {accion}\nGesto: {gesto}\nComando: {comando}", fontsize=14)
    plt.show()

def escena_visual():
    global comando_actual, gesto_actual

    color = (1.0, 1.0, 1.0)  # Blanco
    accion = "Esperando..."

    while True:
        if "cambiar" in comando_actual and gesto_actual == "mano_abierta":
            color = (0.0, 0.0, 1.0)  # Azul
            accion = "Color cambiado a azul"
            comando_actual = ""

        elif "mover" in comando_actual and gesto_actual == "dos_dedos":
            color = (1.0, 0.0, 0.0)  # Rojo
            accion = "Acción mover"
            comando_actual = ""

        else:
            accion = "Esperando..."

        mostrar_accion(color, accion, gesto_actual, comando_actual)
        time.sleep(1)

In [ ]:
hilo_mano = threading.Thread(target=detectar_mano)
hilo_voz = threading.Thread(target=escuchar_comandos)
hilo_escena = threading.Thread(target=escena_visual)

hilo_mano.start()
hilo_voz.start()
hilo_escena.start()

# Esperar a que terminen
hilo_mano.join()
hilo_voz.join()
hilo_escena.join()